In [ ]:
# ============================================
# DBSCAN SOBRE T_final_unsupervised (AUTO/MANUAL)
# Flujo deseado:
#  - AUTO: barre min_samples → calcula eps (k-distance/knee) → evalúa silueta/ruido/clusters SIN graficar
#          → elige el MEJOR → SOLO entonces genera 3 gráficas del mejor modelo:
#              (1) k-distance ("el codo"), (2) tipos (núcleo/borde/ruido), (3) PC1–PC2 coloreado por cluster
#          → imprime métricas inmediatamente → exporta CSV/JSON → crea ZIP con las 3 imágenes
#  - MANUAL: calcula eps (o usa EPS_FIXED), corre DBSCAN, genera SOLO esas 3 gráficas y exporta/ZIP.
# Reglas gráficas: matplotlib puro, 1 figura por imagen, sin seaborn.
# ============================================

import os, json, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score

plt.rcParams["figure.dpi"] = 100

In [ ]:
# -------- CONFIG --------
X = pd.read_csv("T_final_unsupervised.csv")
OUTDIR = Path("dbscan_out"); OUTDIR.mkdir(exist_ok=True)

# --- Selección de modo ---
AUTO_MIN_SAMPLES       = True          # ← pon False para modo manual
MIN_SAMPLES_USER       = 9            # ← usado si AUTO_MIN_SAMPLES=False
EPS_FIXED              = None          # ← si no es None, usa este eps fijo (omite k-distance para selección)

# --- Barrido en modo AUTO ---
MIN_SAMPLES_CANDIDATES = [4, 6, 8, 10, 12, 16, 20]

In [ ]:
# -------- UTILIDADES --------
def _first_two_columns_matrix(Xdf: pd.DataFrame):
    """Devuelve (x1, x2, xlabel, ylabel) usando PC1/PC2 si existen; si no, las dos primeras columnas."""
    cols = list(Xdf.columns)
    if {"PC1", "PC2"}.issubset(cols):
        x1, x2 = Xdf["PC1"].to_numpy(), Xdf["PC2"].to_numpy()
        xlabel, ylabel = "PC1", "PC2"
    else:
        x1, x2 = Xdf.iloc[:, 0].to_numpy(), Xdf.iloc[:, 1].to_numpy()
        xlabel, ylabel = cols[0], cols[1]
    return x1, x2, xlabel, ylabel

def k_distance_values(Xarr: np.ndarray, k: int, algo="kd_tree"):
    """Distancias al k-ésimo vecino más cercano (ordenadas asc)."""
    nbrs = NearestNeighbors(n_neighbors=max(1, k), algorithm=algo).fit(Xarr)
    dists, _ = nbrs.kneighbors(Xarr)
    return np.sort(dists[:, -1])

def knee_from_curve(y: np.ndarray, min_eps=1e-6):
    """
    Triangle method con protección robusta.
    """
    n = len(y)
    if n == 0:
        return 0, min_eps

    x = np.arange(n)
    p1, p2 = np.array([x[0], y[0]]), np.array([x[-1], y[-1]])

    # Si todos los valores son iguales o muy cercanos
    if np.max(y) - np.min(y) < min_eps:
        return n//2, max(y[n//2], min_eps)

    v = p2 - p1
    v_norm = np.linalg.norm(v)
    if v_norm < min_eps:
        return n//2, max(y[n//2], min_eps)

    vn = v / v_norm
    w = np.stack([x, y], axis=1) - p1
    d = np.linalg.norm(w - (w @ vn)[:, None] * vn, axis=1)

    # Evitar extremos
    valid_range = slice(1, -1) if n > 2 else slice(0, n)
    idx = int(np.argmax(d[valid_range]))
    if valid_range.start > 0:
        idx += valid_range.start

    eps_val = max(y[idx], min_eps)
    return idx, eps_val

def run_dbscan(Xarr, eps, min_samples):
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(Xarr)
    labels = db.labels_
    core_mask = np.zeros_like(labels, dtype=bool)
    core_mask[db.core_sample_indices_] = True
    return labels, core_mask

def silhouette_non_noise_or_none(Xarr, labels):
    mask = (labels != -1)
    uniq = np.unique(labels[mask])
    if len(uniq) >= 2 and mask.sum() >= 2:
        return float(silhouette_score(Xarr[mask], labels[mask]))
    return None

# -------- PLOTS (solo se usan para el modelo ganador) --------
def plot_kdistance(kd, k, path, vline_idx=None, hline_eps=None,
                   title="Gráfico k-distance (sugerencia para ε)"):
    plt.figure(figsize=(7.4, 5.2))
    plt.plot(kd, linewidth=1.8)
    if vline_idx is not None:
        plt.axvline(vline_idx, linestyle="--", linewidth=1.8)
    if hline_eps is not None:
        plt.axhline(hline_eps, linestyle="--", linewidth=1.5)
    plt.ylabel(f"Distancia al {k}º vecino"); plt.xlabel("Puntos ordenados")
    plt.title(title)
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight"); plt.show(); plt.close()

def plot_dbscan_types_pc12(Xdf, labels, core_mask, path, title):
    """Núcleo (o), borde (^), ruido (x) por marcadores en PC1–PC2."""
    x1, x2, xl, yl = _first_two_columns_matrix(Xdf)
    noise  = labels == -1
    clust  = labels != -1
    border = clust & (~core_mask)
    core   = clust & core_mask

    plt.figure(figsize=(8.4, 6.2))
    plt.scatter(x1[noise],  x2[noise],  s=18, marker="x", label="Ruido (-1)")
    plt.scatter(x1[border], x2[border], s=26, marker="^", label="Borde")
    plt.scatter(x1[core],   x2[core],   s=26, marker="o", label="Núcleo")
    plt.xlabel(xl); plt.ylabel(yl)
    plt.title(title)
    plt.grid(alpha=0.25); plt.legend(frameon=False)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight"); plt.show(); plt.close()

def plot_pc12_by_cluster(Xdf, labels, path, title):
    """PC1–PC2 coloreado por id de cluster; ruido en 'x'."""
    cols = list(Xdf.columns)
    if {"PC1","PC2"}.issubset(cols):
        x1, x2 = Xdf["PC1"].to_numpy(), Xdf["PC2"].to_numpy()
        xlabel, ylabel = "PC1", "PC2"
    else:
        x1, x2 = Xdf.iloc[:,0].to_numpy(), Xdf.iloc[:,1].to_numpy()
        xlabel, ylabel = cols[0], cols[1]

    uniq = sorted(set(labels) - {-1})
    k = len(uniq)
    cmap = (plt.cm.get_cmap("tab10", k) if k <= 10
            else plt.cm.get_cmap("tab20", k) if k <= 20
            else plt.cm.get_cmap("viridis", k))

    plt.figure(figsize=(8.4, 6.2))

    noise_mask = (labels == -1)
    if noise_mask.any():
        plt.scatter(x1[noise_mask], x2[noise_mask], s=18, marker="x", label="Ruido (-1)")

    for idx, c in enumerate(uniq):
        m = (labels == c)
        plt.scatter(x1[m], x2[m], s=24, label=f"Cluster {c}", c=[cmap(idx)])

    plt.xlabel(xlabel); plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(alpha=0.25); plt.legend(frameon=False, ncol=2)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches="tight"); plt.show(); plt.close()

# -------- PIPELINE PRINCIPAL --------
X_arr = X.to_numpy(dtype=float)

def evaluate_candidate(ms, eps_fixed=None):
    """Evalúa un candidato de min_samples devolviendo métricas y parámetros; sin graficar."""
    k = max(1, ms - 1)
    if eps_fixed is None:
        kd = k_distance_values(X_arr, k=k)
        _, eps_star = knee_from_curve(kd)
    else:
        kd = None
        eps_star = float(eps_fixed)
    labels, core_mask = run_dbscan(X_arr, eps_star, ms)
    n = len(labels)
    noise_ratio = float((labels == -1).sum() / n)
    n_clusters = int(len(set(labels) - {-1}))
    sil = silhouette_non_noise_or_none(X_arr, labels)
    return {
        "min_samples": int(ms),
        "eps_star": float(eps_star),
        "n_clusters": n_clusters,
        "noise_ratio": noise_ratio,
        "silhouette": sil,
        "labels": labels,
        "core_mask": core_mask,
        "k_for_kdist": k
    }

def select_best(results):
    """Selecciona mejor por (silueta desc, -ruido, clusters asc→desc)."""
    def key(r):
        sil = -1.0 if r["silhouette"] is None else r["silhouette"]
        return (sil, -r["noise_ratio"], r["n_clusters"])
    return max(results, key=key)

results = []
if AUTO_MIN_SAMPLES:
    for ms in MIN_SAMPLES_CANDIDATES:
        results.append(evaluate_candidate(ms, eps_fixed=EPS_FIXED))
    best = select_best(results)
else:
    best = evaluate_candidate(int(MIN_SAMPLES_USER), eps_fixed=EPS_FIXED)

# --- Recalcular/obtener insumos del modelo ganador para graficar k-distance ---
best_ms  = best["min_samples"]
best_eps = best["eps_star"]
best_k   = best["k_for_kdist"]

# Si EPS_FIXED es None, el codo es el knee; si EPS_FIXED tiene valor, igualmente calculamos k-distance
# para mostrar "el codo" de referencia y además marcamos eps fijo con línea horizontal.
kd_for_plot = k_distance_values(X_arr, k=best_k)
idx_knee, knee_eps = knee_from_curve(kd_for_plot)

# --- Re-ejecutar DBSCAN ganador (para asegurar coherencia en figuras) ---
labels, core_mask = run_dbscan(X_arr, best_eps, best_ms)

# -------- FIGURAS (solo del modelo ganador) --------
png_knee   = OUTDIR / "kdistance_best.png"
png_types  = OUTDIR / "dbscan_types_best.png"
png_clust  = OUTDIR / "dbscan_clusters_best.png"

In [ ]:
plot_kdistance(
    kd_for_plot, best_k, png_knee,
    vline_idx=idx_knee,
    hline_eps=(best_eps if EPS_FIXED is not None else None),
    title=f"Gráfico k-distance (k={best_k})"
)

In [ ]:
plot_dbscan_types_pc12(
    X, labels, core_mask, png_types,
    title=f"DBSCAN — Tipos (ε={best_eps:.3f}, min_samples={best_ms})"
)

In [ ]:
plot_pc12_by_cluster(
    X, labels, png_clust,
    title=f"DBSCAN — PC1–PC2 por cluster (ε={best_eps:.3f}, min_samples={best_ms})"
)

In [ ]:
# -------- MÉTRICAS --------
n_points = len(labels)
noise_ratio = float((labels == -1).sum() / n_points)
n_clusters = int(len(set(labels) - {-1}))
sil = silhouette_non_noise_or_none(X_arr, labels)

print("=== MODELO GANADOR ===")
print("min_samples* :", best_ms)
print("eps*         :", round(best_eps, 6))
print("clusters     :", n_clusters)
print("ruido (%)    :", round(100*noise_ratio, 2))
print("silueta      :", "NA" if sil is None else round(sil, 6))

In [ ]:
# -------- EXPORTS --------
# Dataset con etiquetas
out_df = X.copy()
out_df["label_dbscan"] = labels
out_df_path = OUTDIR / "T_final_unsupervised_dbscan.csv"
out_df.to_csv(out_df_path, index=False)

# Auxiliares + métricas
pd.DataFrame({"label_dbscan": labels}).to_csv(OUTDIR / "labels.csv", index=False)

metrics = {
    "mode_auto_min_samples": bool(AUTO_MIN_SAMPLES),
    "chosen_min_samples": int(best_ms),
    "chosen_eps": float(best_eps),
    "k_for_kdistance": int(best_k),
    "n_points": int(n_points),
    "n_clusters_excl_noise": int(n_clusters),
    "noise_ratio": float(noise_ratio),
    "silhouette_non_noise": (None if sil is None else float(sil)),
}
with open(OUTDIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

config = {
    "input_csv": "T_final_unsupervised.csv",
    "auto_min_samples": bool(AUTO_MIN_SAMPLES),
    "min_samples_user": (None if AUTO_MIN_SAMPLES else int(MIN_SAMPLES_USER)),
    "eps_fixed": (None if EPS_FIXED is None else float(EPS_FIXED)),
    "min_samples_candidates": list(map(int, MIN_SAMPLES_CANDIDATES)),
    "columns_used": X.columns.tolist()
}
with open(OUTDIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

In [ ]:
# -------- BUNDLE  --------
zip_path = OUTDIR / "dbscan_bundle.zip"
to_zip = [
    out_df_path,
    OUTDIR / "labels.csv",
    OUTDIR / "metrics.json",
    OUTDIR / "config.json",
    png_knee, png_types, png_clust
]
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in to_zip:
        if isinstance(f, Path): f = f
        if os.path.exists(f):
            zf.write(f, arcname=Path(f).name)

print("ZIP:", zip_path.resolve())
